# Graphql

> Fill in a module description here

In [1]:
# | default_exp graphql

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [3]:
# | export
import typing
import strawberry

MVP Spec:
-        save_button = gr.Button("Save Files and Transcriptions")
            save_button.click(
                save_files_and_transcriptions,
                inputs=[user_id_input, user_name_input, file_input],
                outputs=output_text,


        with gr.TabItem("Query and Generate Video"):
            gr.Markdown("## Query and Generate Video")
            gr.Markdown(
                "Enter a query to find relevant utterances and generate a video from them. Use the transcript selector above."
            )
            query_input = gr.Textbox(label="Enter Query")
            select_all_checkbox = gr.Checkbox(label="Select All", value=False)
            user_ids_input_2 = gr.CheckboxGroup(
                label="User IDs",
                choices=get_user_names_and_transcript_counts(db),
                interactive=True,
            )
            select_all_checkbox.change(
                fn=update_user_selection,
                inputs=[select_all_checkbox],
                outputs=[user_ids_input_2],
            )
            fetch_button = gr.Button("Create Video")
            video_download = gr.File(label="Download Video", interactive=False)
            fetch_button.click(
                fetch_utterances,
                inputs=[query_input, user_ids_input_2],
                outputs=video_download,
            )

- Screen 1
    - add user name + metadata +files
    - get back confirmation of file upload

- Screen 2
    - Enter query
    - Get back list of Utterances
    - (frontend only: Select Utterances/words, pick custom times based on word timing) -> UtteranceSegments
    - Input UtteranceSegments, gets back Video with Pending result (+ page redirect)

- Screen 3
    - Video screen - looks at duration and polls more frequentlyas it gets closer to end.


Graphql spec - claude thinks it should look something like this:

```
type Mutation {
  saveFilesAndTranscriptions(userId: ID!, userName: String!, files: [Upload!]!): String!
  createVideo(query: String!, userIds: [ID!]!): String!
}

type Query {
  getUserNamesAndTranscriptCounts: [UserOption!]!
  getRelevantUtterancesFromQuery(query: String!, transcriptIds: [ID!]!): [Utterance!]!
}

type UserOption {
  label: String!
  value: ID!
}

type Utterance {
  id: ID!
  confidence: Float!
  end: Int!
  speaker: String
  start: Int!
  text: String!
  words: [Word!]!
}

type Word {
  id: ID!
  confidence: Float!
  end: Int!
  speaker: String
  start: Int!
  text: String!
}
```

In [ ]:
#| export
import typing
import strawberry
from strawberry.file_uploads import Upload
from uuid import UUID

@strawberry.type
class UserWithTranscriptCount:
    name: str
    id: strawberry.ID
    transcript_count: int

@strawberry.type
class Word:
    id: strawberry.ID
    confidence: float
    end: int
    speaker: str
    start: int
    text: str

@strawberry.type
class Utterance:
    id: strawberry.ID
    confidence: float
    end: int
    speaker: str
    start: int
    text: str
    words: typing.List[Word]

@strawberry.type
class VideoJob:
    id: strawberry.ID
    status: str
    video_url: str

@strawberry.type
class Query:
    @strawberry.field
    def get_user_names_and_transcript_counts(self) -> typing.List[UserWithTranscriptCount]:
        # Dummy resolver
        return [UserWithTranscriptCount(name="User 1", id=strawberry.ID("1"), transcript_count=2)]

    @strawberry.field
    def get_relevant_utterances_from_query(self, query: str, transcript_ids: typing.List[strawberry.ID]) -> typing.List[Utterance]:
        # Dummy resolver
        return [Utterance(id=strawberry.ID("1"), confidence=0.9, end=1000, speaker="Speaker 1", start=0, text="Hello", words=[])]

@strawberry.type
class Mutation:
    @strawberry.mutation
    def save_files_and_transcriptions(self, user_id: strawberry.ID, user_name: str, files: typing.List[Upload]) -> str:
        # Dummy resolver
        return "2 transcriptions saved."

    @strawberry.mutation
    def create_video(self, query: str, user_ids: typing.List[strawberry.ID]) -> VideoJob:
        # Dummy resolver
        return VideoJob(id=strawberry.ID("1"), status="pending", video_url="")

schema = strawberry.Schema(query=Query, mutation=Mutation)

test queries

```

mutation {
  createVideo(query: "example query", userIds: ["1", "2"]) {
    id
    status
    videoUrl
  }
}

mutation($files: [Upload!]!) {
  saveFilesAndTranscriptions(userId: "123", userName: "John Doe", files: $files)
}

query {
  getRelevantUtterancesFromQuery(query: "example query", transcriptIds: ["1", "2"]) {
    id
    confidence
    end
    speaker
    start
    text
    words {
      id
      confidence
      end
      speaker
      start
      text
    }
  }
}


query {
  getUserNamesAndTranscriptCounts {
    name
    id
    transcriptCount
  }
}
```

In [8]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore